# EP.0 Transformer：从张量形状到完整 Encoder–Decoder

配套文章：[Transformer 的数学表示与代码实现](https://momoyeyu.github.io/posts/llm-transformer/)。文章负责完整数学推导；本 Notebook 负责把每个公式落成可执行代码，并逐步核对形状、数值与边界。这里首次出现的实现会直接展开真实源码，而不是隐藏在导入中。

学习路线：缩放点积注意力 → 多头 → FFN → mask → embedding/位置 → Encoder/Decoder → 完整 Transformer。下一章见 [01_moe.ipynb](01_moe.ipynb)。

In [1]:
from pathlib import Path
import random
import sys

root = Path.cwd()
if not (root / "slm").is_dir():
    root = next((parent for parent in root.parents if (parent / "slm").is_dir()), None)
if root is None:
    raise RuntimeError("请从仓库根目录或 notebooks 目录运行")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(1)

## 可复用实验工具

先展开随机种子与统一输出函数。后续章节会把它们当作已经学过的公共工具导入。

In [2]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)

In [3]:
def show(name: str, value) -> None:
    print(f"{name:<28} {value}")

In [4]:
set_seed(0)
show("Python", sys.version.split()[0])
show("PyTorch", torch.__version__)
show("device", "cpu")

Python                       3.12.12
PyTorch                      2.14.1
device                       cpu


## 1. 缩放点积注意力：为什么除以 $\sqrt{D_k}$？

给定 $Q\in\mathbb{R}^{B\times N\times D_k}$、$K\in\mathbb{R}^{B\times M\times D_k}$、$V\in\mathbb{R}^{B\times M\times D_v}$：

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}(QK^\top/\sqrt{D_k})V.$$

softmax 必须沿最后一个 key 轴 $M$ 做；输出形状为 $[B,N,D_v]$。缩放用于控制随机点积随维度增长的量级。

In [5]:
def attention(
    query: torch.Tensor,             # [batch, n, d_k]
    key: torch.Tensor,               # [batch, n, d_k]
    value: torch.Tensor,             # [batch, n, d_v]
    mask: torch.Tensor | None = None # [n, n] or [batch, n, n]
) -> torch.Tensor:
    d_k = query.size(-1)
    scores = query @ key.transpose(-2, -1) / math.sqrt(d_k)  # [batch, n, n]
    if mask is not None:
        scores = scores + mask.to(scores.dtype)
    weights = F.softmax(scores, dim=-1)      # [batch, n, n]
    return weights @ value  # [batch, n, d_v]

In [6]:
import math


import torch
import torch.nn.functional as F


set_seed(0)

for d_k in [64, 512, 4096]:
    q, k = torch.randn(1, 32, d_k), torch.randn(1, 32, d_k)
    scores = q @ k.transpose(-2, -1)
    show(f"d_k={d_k:<5} logits std", f"{scores.std():.1f} unscaled -> {(scores / math.sqrt(d_k)).std():.3f} scaled")

q, k, v = torch.randn(2, 8, 16), torch.randn(2, 8, 16), torch.randn(2, 8, 16)
weights = F.softmax(q @ k.transpose(-2, -1) / 4, dim=-1)
show("weights row sums", weights.sum(-1)[0].tolist())
show("attention output", tuple(attention(q, k, v).shape))

d_k=64    logits std         7.9 unscaled -> 0.993 scaled
d_k=512   logits std         22.9 unscaled -> 1.014 scaled
d_k=4096  logits std         65.2 unscaled -> 1.019 scaled
weights row sums             [1.0, 0.9999999403953552, 1.0, 1.0, 1.0, 1.0000001192092896, 1.0000001192092896, 1.0]
attention output             (2, 8, 16)


输出应显示：未缩放 logits 的标准差随 $\sqrt{D_k}$ 增长，缩放后接近同一量级；每行权重和为 1。它是初始化分布下的量级检查，不是对所有训练状态的界。

## 2. 多头注意力：分头不等于增加总投影矩阵

每个头把 $D_{model}$ 投影到 $D_k=D_v=D_{model}/H$，拼接后再经输出投影。合起来仍对应 Q/K/V/O 四个 $D_{model}\times D_{model}$ 矩阵，因此在这个实现里改变头数不会改变这些矩阵的总参数量或主矩阵乘 MAC；这不意味着头数不影响表达或准确率。

In [7]:
class MHA(nn.Module):
    def __init__(self, d_model: int, num_heads: int) -> None:
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = self.d_model // self.num_heads
        self.d_v = self.d_model // self.num_heads  # d_v could be simplified as d_k here, but I want to keep it clear
        self.w_q = nn.ModuleList([nn.Linear(self.d_model, self.d_k, bias=False) for _ in range(self.num_heads)])
        self.w_k = nn.ModuleList([nn.Linear(self.d_model, self.d_k, bias=False) for _ in range(self.num_heads)])
        self.w_v = nn.ModuleList([nn.Linear(self.d_model, self.d_v, bias=False) for _ in range(self.num_heads)])
        self.w_o = nn.Linear(self.d_v * self.num_heads, self.d_model, bias=False)

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        # query/key/value: [batch, n, d_model]
        heads = [attention(self.w_q[i](query), self.w_k[i](key), self.w_v[i](value), mask) for i in range(self.num_heads)]  # h × [batch, n, d_v]
        return self.w_o(torch.cat(heads, dim=-1))  # cat: [batch, n, h·d_v] -> [batch, n, d_model]

In [8]:
import torch


set_seed(0)
d_model, num_heads = 16, 4
mha = MHA(d_model, num_heads)
x = torch.randn(2, 8, d_model)

out = mha(x, x, x)
show("MHA output", tuple(out.shape))

heads = [attention(mha.w_q[i](x), mha.w_k[i](x), mha.w_v[i](x)) for i in range(num_heads)]
show("per-head concat matches", torch.allclose(mha.w_o(torch.cat(heads, dim=-1)), out, atol=1e-6))

# d_k = d_v = d_model/h，参数量与 h 无关，消融时突出头数本身的作用
show("param count", f"{sum(p.numel() for p in mha.parameters())} = 4·d_model² = {4 * d_model * d_model}")

MHA output                   (2, 8, 16)
per-head concat matches      True
param count                  1024 = 4·d_model² = 1024


演示把每个头单独计算后拼接，结果与 `MHA.forward` 一致；输出形状恢复为 `[B,N,D_model]`。

## 3. FFN：逐 token 的非线性通道变换

注意力在 token 轴混合信息；FFN 对每个 token 独立复用同一 MLP，在通道轴做 $D_{model}\to D_{ff}\to D_{model}$。默认 ReLU 保持本章 Transformer 行为，后续 MoE 会复用同一类并换成 GELU。

In [9]:
class FFN(nn.Module):
    def __init__(self, d_model: int, d_ff: int, activation: type[nn.Module] = nn.ReLU) -> None:
        super().__init__()
        self.d_model = d_model
        self.d_ff = d_ff
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            activation(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, n, d_model]
        return self.mlp(x)

In [10]:
x = torch.randn(2, 5, 16)
show('FFN shape', tuple(FFN(16, 32)(x).shape))

FFN shape                    (2, 5, 16)


输入输出都是 `[2,5,16]`：FFN 不改变序列长度，也不在不同 token 间通信；非线性使它不退化为单个线性层。

## 4. Mask：把不可见位置变成 $-\infty$

causal mask 阻止位置 $i$ 读取未来 $j>i$；padding mask 阻止读取补齐 key。加到 logits 后，softmax 把这些位置变成 0。实现假设每个参与评估的 query 至少有一个可见 key；整行全为 $-\infty$ 会使 softmax 无定义。

In [11]:
_NEG_INF = float("-inf")

In [12]:
def causal_mask(size: int, device: torch.device | None = None) -> torch.Tensor:
    i = torch.arange(size, device=device).unsqueeze(1)      # [n, 1]
    j = torch.arange(size, device=device).unsqueeze(0)      # [1, n]
    return torch.where(j > i, _NEG_INF, 0.0)                # [n, n]

In [13]:
def padding_mask(pad: torch.Tensor) -> torch.Tensor:
    # pad: [batch, n]，1 表示 padding
    return torch.where(pad.bool(), _NEG_INF, 0.0).unsqueeze(1)  # [batch, 1, n]

In [14]:
import torch
import torch.nn.functional as F


set_seed(0)
n = 6
q, k, v = torch.randn(1, n, 8), torch.randn(1, n, 8), torch.randn(1, n, 8)
scores = q @ k.transpose(-2, -1) / (q.size(-1) ** 0.5)

show("causal mask", causal_mask(n))
weights = F.softmax(scores + causal_mask(n), dim=-1)
show("row-0 weights (self only)", weights[0, 0].tolist())

# 改动未来 token 不影响过去位置的输出
k2, v2 = k.clone(), v.clone()
k2[:, -1], v2[:, -1] = 99.0, -99.0
out, out2 = attention(q, k, v, causal_mask(n)), attention(q, k2, v2, causal_mask(n))
show("past rows unchanged", torch.allclose(out[:, :-1], out2[:, :-1]))

pad = torch.zeros(1, n, dtype=torch.long)
pad[0, -2:] = 1
weights = F.softmax(scores + padding_mask(pad), dim=-1)
show("pad keys' max weight", weights[..., -2:].abs().max().item())

causal mask                  tensor([[0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0.]])
row-0 weights (self only)    [1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
past rows unchanged          True
pad keys' max weight         0.0


因果矩阵第一行只能看自己；修改最后一个 key/value 不会影响更早 query。padding key 的最大权重为 0。

## 5. Token、位置与分类头

`Embedding` 把 id `[B,N]` 变为 `[B,N,D]` 并乘 $\sqrt D$。正弦位置编码交替使用 $\sin$/$\cos$，与 token embedding 相加。`LinearClassifier` 把每个位置映射回词表；传入 embedding 权重即可 weight tying。

In [15]:
class Embedding(nn.Module):
    def __init__(self, vocab_size: int, d_model: int) -> None:
        super().__init__()
        self.d_model = d_model
        self.w_e = nn.Embedding(vocab_size, d_model)
        nn.init.normal_(self.w_e.weight, mean=0.0, std=d_model ** -0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, n]（token id）
        return self.w_e(x) * math.sqrt(self.d_model)  # [batch, n, d_model]

In [16]:
class PositionEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000) -> None:
        super().__init__()
        assert d_model % 2 == 0
        self.d_model = d_model
        self.max_len = max_len  # cache length
        self.register_buffer("cache", self.encode(0, self.max_len), persistent=False)

    def encode(self, start: int, end: int, device: torch.device | None = None) -> torch.Tensor:
        n = end - start
        pos = torch.arange(start, end, dtype=torch.float32, device=device).unsqueeze(-1)  # [n, 1]
        i = torch.arange(0, self.d_model, 2, dtype=torch.float32, device=device).unsqueeze(0)  # [1, d_model / 2]
        div = torch.exp(-math.log(10000) * i / self.d_model)  # [1, d_model / 2]
        position_encoding = torch.zeros(n, self.d_model, device=device)  # [n, d_model]
        position_encoding[:, 0::2] = torch.sin(pos * div)   # [n,1]×[1,d_model/2] -> [n, d_model/2]
        position_encoding[:, 1::2] = torch.cos(pos * div)   # [n, d_model/2]
        return position_encoding

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, n, d_model]
        n = x.size(-2)
        if n <= self.max_len:
            return x + self.cache[:n].to(x.dtype)  # cache[:n]: [n, d_model]
        pos_encoding = torch.zeros_like(x)                                          # [batch, n, d_model]
        pos_encoding[:, :self.max_len] = self.cache.to(x.dtype)                     # [batch, max_len, d_model]
        pos_encoding[:, self.max_len:] = self.encode(self.max_len, n, device=x.device).to(x.dtype)  # [batch, n-max_len, d_model]
        return x + pos_encoding

In [17]:
class LinearClassifier(nn.Module):
    def __init__(self, d_model: int, vocab_size: int, weight: torch.Tensor | None = None) -> None:
        super().__init__()
        self.classifier = nn.Linear(d_model, vocab_size, bias=False)
        if weight is not None:
            self.classifier.weight = weight

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(x)  # [batch, n, d_model] -> [batch, n, vocab_size]

In [18]:
pe = PositionEncoding(16, max_len=8)
show('PE first 3 rows', pe(torch.zeros(1, 3, 16))[0])
emb = Embedding(20, 16)
head = LinearClassifier(16, 20, emb.w_e.weight)
show('tied weight identity', head.classifier.weight is emb.w_e.weight)

PE first 3 rows              tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  3.1098e-01,  9.5042e-01,  9.9833e-02,
          9.9500e-01,  3.1618e-02,  9.9950e-01,  9.9998e-03,  9.9995e-01,
          3.1623e-03,  9.9999e-01,  1.0000e-03,  1.0000e+00,  3.1623e-04,
          1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  5.9113e-01,  8.0658e-01,  1.9867e-01,
          9.8007e-01,  6.3203e-02,  9.9800e-01,  1.9999e-02,  9.9980e-01,
          6.3245e-03,  9.9998e-01,  2.0000e-03,  1.0000e+00,  6.3246e-04,
          1.0000e+00]])
tied weight identity         True


前三行直接显示位置 0/1/2 的 sin/cos 模式。缓存之外仍可按公式计算位置，但“能计算”不等于模型在未训练长度上必然泛化。

## 6. Encoder：self-attention + FFN

每个子层采用残差后 LayerNorm：$x\leftarrow\mathrm{LN}(x+F(x))$。这里把 `nn.LayerNorm` 当作 PyTorch 基元；其内部统计量会在 [02_normalization.ipynb](02_normalization.ipynb) 从零实现。

In [19]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.mha = MHA(d_model, num_heads)
        self.ln1 = nn.LayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        # x: [batch, src_len, d_model]，各子层均保持此形状
        residual = x
        x = self.mha(x, x, x, mask)
        x = self.dropout(x)
        x = self.ln1(x + residual)
        residual = x
        x = self.ffn(x)
        x = self.dropout(x)
        x = self.ln2(x + residual)
        return x

In [20]:
class Encoder(nn.Module):
    def __init__(self, num_layers: int, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        # x: [batch, src_len, d_model]
        for block in self.blocks:
            x = block(x, mask)
        return x

Encoder 的 self-attention 使用同一序列作 Q/K/V；每层保持 `[B,src_len,D]`，mask 可广播到每个 query。

## 7. Decoder：masked self-attention + cross-attention

Decoder 先在目标序列上做 causal self-attention，再做 cross-attention：Q 来自目标，K/V 来自 Encoder。因而 `tgt_len` 与 `src_len` 可以不同，cross logits 形状是 `[B,tgt_len,src_len]`。

In [21]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.masked_mha = MHA(d_model, num_heads)
        self.ln1 = nn.LayerNorm(d_model)
        self.cross_mha = MHA(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.ln3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, y: torch.Tensor, h: torch.Tensor, self_mask: torch.Tensor | None = None, cross_mask: torch.Tensor | None = None) -> torch.Tensor:
        # y: [batch, tgt_len, d_model], h: [batch, src_len, d_model]
        residual = y
        y = self.masked_mha(y, y, y, self_mask)
        y = self.dropout(y)
        y = self.ln1(y + residual)
        residual = y
        y = self.cross_mha(y, h, h, cross_mask)
        y = self.dropout(y)
        y = self.ln2(y + residual)
        residual = y
        y = self.ffn(y)
        y = self.dropout(y)
        y = self.ln3(y + residual)
        return y

In [22]:
class Decoder(nn.Module):
    def __init__(self, num_layers: int, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.blocks = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

    def forward(self, y: torch.Tensor, h: torch.Tensor, self_mask: torch.Tensor | None = None, cross_mask: torch.Tensor | None = None) -> torch.Tensor:
        # y: [batch, tgt_len, d_model], h: [batch, src_len, d_model]
        for block in self.blocks:
            y = block(y, h, self_mask, cross_mask)
        return y

三个残差子层依次是目标自注意力、源目标交叉注意力、逐 token FFN；每个阶段都回到目标形状。

## 8. 完整 Transformer 与 teacher forcing

完整模型组合 embedding、位置、Encoder、Decoder 和 tied 分类头。训练时目标序列整体输入，但 causal mask 保证位置 $t$ 只能使用不晚于 $t$ 的输入；`logits[:,t]` 对齐下一个目标 `tgt[:,t+1]`。padding 标签必须从 loss 中忽略。

In [23]:
class Transformer(nn.Module):
    def __init__(
            self,
            d_model: int,
            max_len: int,
            num_layers: int,
            num_heads: int,
            d_ff: int,
            vocab_size: int,
            dropout: float = 0.1
    ) -> None:
        super().__init__()
        self.embedding = Embedding(vocab_size, d_model)
        self.pos_encoder = PositionEncoding(d_model, max_len)
        self.dropout = nn.Dropout(dropout)
        self.encoder = Encoder(num_layers, d_model, num_heads, d_ff, dropout)
        self.decoder = Decoder(num_layers, d_model, num_heads, d_ff, dropout)
        self.classifier = LinearClassifier(d_model, vocab_size, self.embedding.w_e.weight)

    def forward(
        self,
        src: torch.Tensor,                    # [batch, src_len]
        tgt: torch.Tensor,                    # [batch, tgt_len]
        src_pad: torch.Tensor | None = None,  # [batch, src_len], 1 表示 padding
        tgt_pad: torch.Tensor | None = None,  # [batch, tgt_len], 1 表示 padding
    ) -> torch.Tensor:
        src = self.dropout(self.pos_encoder(self.embedding(src)))  # [batch, src_len] -> [batch, src_len, d_model]
        tgt = self.dropout(self.pos_encoder(self.embedding(tgt)))  # [batch, tgt_len] -> [batch, tgt_len, d_model]

        src_mask = padding_mask(src_pad) if src_pad is not None else None  # [batch, 1, src_len]

        causal = causal_mask(tgt.size(1), device=tgt.device)  # [tgt_len, tgt_len]
        if tgt_pad is not None:
            tgt_mask = causal + padding_mask(tgt_pad)  # [batch, tgt_len, tgt_len]
        else:
            tgt_mask = causal

        h = self.encoder(src, src_mask)                 # [batch, src_len, d_model]
        z = self.decoder(tgt, h, tgt_mask, src_mask)    # [batch, tgt_len, d_model]
        return self.classifier(z)                       # logits: [batch, tgt_len, vocab_size]

In [24]:
import torch
import torch.nn.functional as F


set_seed(0)
model = Transformer(d_model=64, max_len=32, num_layers=2, num_heads=4, d_ff=128, vocab_size=50)

show("weight tying", model.classifier.classifier.weight is model.embedding.w_e.weight)

# teacher forcing：整段目标序列一次喂入，causal mask 挡住未来
src = torch.randint(0, 50, (2, 10))
tgt = torch.randint(0, 50, (2, 7))
tgt_pad = torch.zeros(2, 7, dtype=torch.long)
tgt_pad[1, 5:] = 1
logits = model(src, tgt, tgt_pad=tgt_pad)
show("logits", tuple(logits.shape))

# 训练对下一 token 算交叉熵，并忽略 padding 目标；推理取 argmax 即 greedy
labels = tgt[:, 1:].masked_fill(tgt_pad[:, 1:].bool(), -100)
loss = F.cross_entropy(logits[:, :-1].reshape(-1, 50), labels.reshape(-1), ignore_index=-100)
show("teacher forcing loss", round(loss.item(), 3))
show("greedy next token", logits[0, -1].argmax().item())

weight tying                 True
logits                       (2, 7, 50)
teacher forcing loss         5.917
greedy next token            28


演示只对随机初始化 logits 计算一次 teacher-forcing loss，并不代表模型已经学会翻译。输出同时验证 tied 权重身份、logits 形状与 padding-aware 标签移位。